# **Problem Statement**

## **Business Context**

Aeolus Renewables is an independent power producer operating a fleet of 1,150 onshore wind turbines (2.5 MW class) across 16 wind farms in the central plains region, with a combined installed capacity of roughly 2.87 GW. The company sells the electricity its turbines produce to the grid under long-term power purchase agreements, where revenue is tied directly to how much energy is delivered - so every hour a turbine is offline is energy that cannot be sold. A regional operations centre monitors the fleet around the clock, supported by an O&M organisation of about 140 field technicians.

Each turbine continuously streams condition data through its SCADA (Supervisory Control and Data Acquisition) system - drivetrain vibration, bearing and oil temperatures, rotor and generator speed, and power output - recorded as 10-minute averages, the standard logging resolution for utility-scale turbines. The exact channels vary by turbine type, but together they describe the health of the main subassemblies in near-real time. Today, maintenance runs on a mix of fixed-interval inspections and reactive repair: technicians follow a scheduled service calendar, and the control room responds when an automated alarm trips or a turbine faults offline.

The most consequential components are in the drivetrain - the gearbox and main bearing - which are expensive, slow to procure, and require a mobile crane to replace. The gearbox alone represents approximately 13% of the overall capital cost of an onshore turbine, and within gearboxes the failures are dominated by bearings: one widely-cited breakdown puts the split at bearings (70%), gears (26%) and other causes (4%). The core problem is recognition. By the time a drivetrain fault has developed far enough to matter, its signature is real but still tangled in normal operating noise across many channels - and the existing fixed thresholds only trip once the fault is near-catastrophic, when the turbine is already offline or the component has seized. A genuinely fault drivetrain can run for hours or days looking "normal" to a threshold alarm, while a control-room analyst has no practical way to tell it apart from a healthy machine reacting to gusty wind. The consequences:

- Each unplanned drivetrain failure takes a turbine offline for an estimated 7–21 days, directly forfeiting saleable energy under the power purchase agreement.
- Emergency crane mobilisation and expedited parts run at a steep premium over the same work scheduled in advance, making an unplanned gearbox replacement a major cost event.
- A degrading main bearing left running frequently destroys the gearbox it feeds, converting a contained repair into a far larger one.
- Control-room analysts manually scan a flood of channels across 1,150 turbines, and alarm fatigue means genuine degradation signals slip through unnoticed until the machine faults offline.


## **Objective**

This proof of concept builds a drivetrain-condition classifier that reads each turbine's live sensor signature and labels the drivetrain as "fault" or "normal", serving operations-centre analysts and maintenance planners. The solution

- Reads each turbine's multi-channel sensor signature and produces a clear fault-vs-normal signal, so analysts can concentrate on the handful of machines genuinely in a fault condition rather than scanning the whole fleet.
- Distinguishes a truly fault drivetrain from normal operating noise more reliably than fixed thresholds, catching faults that are present but not yet severe enough to trip a catastrophic alarm - the window in which a turbine can still be stopped before a bearing fault cascades into gearbox destruction.
Is deliberately tuned to favour catching true failures over avoiding false alerts, because a missed fault drivetrain costs far more than an unnecessary inspection.
- Establishes a measurable detection baseline on historical fleet data, so the capability's accuracy and its operational value can be judged on evidence before any wider rollout.

Once proven at proof-of-concept scale, this capability would give Aeolus a defensible basis to move drivetrain maintenance from reactive repair toward condition-based intervention - reducing unplanned downtime, protecting saleable energy revenue, and containing repairs before they escalate across a 1150-turbine fleet.

## **Data Dictionary**

The dataset, `wind_turbine_detection.csv` contains 10-minute SCADA records for 15 turbines.

### Identifiers & Metadata

| Column | Data Type | Description |
| --- | --- | --- |
| timestamp | datetime | Date and time of the 10-minute logging interval; establishes chronological sequence. |
| turbine_id | object (categorical) | Unique code identifying individual wind turbines; used to track specific asset history. |

### Environmental Conditions

| Column | Data Type | Description |
| --- | --- | --- |
| rated_power_kW | float | Maximum engineered power capacity of the turbine; defines its performance baseline. |
| wind_speed_mps | float | Velocity of the incoming wind; the primary driver of kinetic energy input. |
| wind_direction_deg | float | Compass direction of oncoming wind; used to assess turbine alignment. |
| turbulence_intensity | float | Measure of wind speed fluctuation; higher intensity increases structural fatigue. |
| air_density_kgm3 | float | Mass of air per unit volume; directly impacts aerodynamic lift and power potential. |
| ambient_temp_C | float | Outdoor air temperature surrounding the turbine; affects cooling efficiency. |
| humidity_pct | float | Relative moisture level in the air; flags risks for electrical insulation degradation or corrosion. |

### Operational Control & State

| Column | Data Type | Description |
| --- | --- | --- |
| power_output_kW | float | Real-time electricity generated; drops or fluctuations can signal mechanical drag. |
| rotor_speed_rpm | float | Rotational speed of the main blades; reflects low-speed shaft dynamics. |
| generator_speed_rpm | float | Rotational speed of the generator shaft; crucial for detecting gearbox slip. |
| blade_pitch_angle_deg | float | Angle of the blades relative to the wind; adjusted to control power and rotor speed. |
| yaw_misalignment_deg | float | Angle deviation between wind direction and nacelle orientation; high values cause uneven drivetrain stress. |

### Thermal Metrics (Component Health)

| Column | Data Type | Description |
| --- | --- | --- |
| gearbox_oil_temp_C | float | Temperature of the lubricating oil; spikes indicate excessive mechanical friction. |
| gearbox_bearing_temp_C | float | Internal temperature of gearbox bearings; a leading indicator of bearing wear. |
| generator_bearing_temp_C | float | Temperature of generator bearings; flags alignment or lubrication issues. |
| generator_winding_temp_C | float | Temperature of internal electrical coils; spikes indicate electrical overload or cooling failure. |
| main_bearing_temp_C | float | Temperature of the primary low-speed shaft bearing; handles massive structural loads. |
| nacelle_temp_C | float | Air temperature inside the enclosed housing; reflects global internal heat dissipation. |

### Vibration & Diagnostics (FFT)

| Column | Data Type | Description |
| --- | --- | --- |
| drivetrain_vibration_rms_mmps | float | Overall energy of drivetrain vibrations; general indicator of mechanical roughness. |
| tower_vibration_mmps | float | Structural oscillation of the turbine tower; flags aerodynamic or rotor imbalance. |
| vib_fft_bearing_bpfo | float | Vibration amplitude at the bearing outer-race defect frequency; tracks outer-ring pitting. |
| vib_fft_bearing_bpfi | float | Vibration amplitude at the bearing inner-race defect frequency; tracks inner-ring pitting. |
| vib_fft_gearmesh | float | Vibration amplitude at the teeth-meshing frequency; isolates gearbox tooth wear or misalignment. |
| vib_fft_sideband | float | Vibration amplitude surrounding main frequencies; flags localized faults like cracked gear teeth. |

### Lubrication & Particle Analysis

| Column | Data Type | Description |
| --- | --- | --- |
| oil_particle_count | float | Quantity of metallic debris suspended in lubricating oil; direct indicator of component wear. |
| oil_pressure_bar | float | Pressure of the lubrication system; drops indicate leaks, pump failures, or oil thinning. |

### Asset Lifecycle & History

| Column | Data Type | Description |
| --- | --- | --- |
| operating_hours_total | float | Cumulative runtime of the turbine; represents the asset's total mechanical mileage. |
| cumulative_energy_MWh | float | Total historical electricity produced; measures the lifetime work done by the drivetrain. |
| load_cycles | int | Total count of fatigue-inducing stress variations; correlates directly with structural aging. |
| hours_since_last_maintenance | float | Time elapsed since last service; crucial for identifying maintenance-cycle fatigue. |
| prior_fault_count | int | Total historical fault events triggered; highlights chronic or poorly repaired issues. |
| component_age_days | float | Elapsed lifetime of active components; captures chronological wear independent of runtime. |

### Targets (Labels)

| Column | Data Type | Description |
| --- | --- | --- |
| failure | int (0/1) | The target variable; boolean flag indicating if the drivetrain or turbine is healthy (0) or failing (1). |

# **Please read the instructions carefully before starting the project.**

This is a commented Python notebook file in which all the instructions and tasks to be performed are mentioned.

* A significant portion of the code required to conduct the analysis and build and evaluate the predictive models are pre-written in the notebook, and only need to be executed.
* In certain sections, blanks '\_\_\_\_\_' are provided in the notebook that
needs to be filled appropriately to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* In certain sections, mutliple lines of code are provided and commented out. The appropriate code snippet needs to be uncommented before executing the respective code cells.
* Identify the task to be performed correctly, and only then proceed to fill in the blank or uncomment code snippets.
* Sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in the presentation and submit the same. Any mathematical or computational details that are a graded part of the project can be included in the Appendix section of the presentation.

# **Installing and Importing the Necessary Libraries**

In [ ]:
!pip install pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 tensorflow==2.20.0 keras==3.13.2 xgboost==3.2.0 seaborn==0.13.2 matplotlib==3.10.0 -q

**Note**:
- After running the above cell, kindly restart the notebook kernel (for VS Code) or runtime (for Google Colab), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Standard libraries for tracking execution time and vector/matrix operations
import time
import numpy as np

# Data manipulation and visualization libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tree-based and ensemble machine learning classifiers
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Utilities for handling class imbalance, model evaluation, and metric calculations
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

# Deep learning framework and specific layers for building Artificial Neural Networks (ANNs)
import tensorflow as tf
from keras.models import Sequential  # Model for building NN sequentially.
from keras.layers import Dense, Dropout, BatchNormalization

# Preprocessing tools for scaling data and tools for model optimization/explainability
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV  # To tune different models
from sklearn.inspection import permutation_importance

# Configuration settings to suppress warnings and format data display outputs
import warnings
warnings.filterwarnings('ignore')              # Suppress warnings for cleaner output logs
sns.set_style('whitegrid')                     # Set a consistent clean grid style for plots
pd.set_option('display.max_columns', None)     # Prevent truncation of columns when displaying dataframes

# **Loading the Data**

In [ ]:
# uncomment and run the below code snippets if you're using Google Colab and the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('wind_turbine_detection.csv')

# **Data Overview**

## Viewing the first and last 5 rows of the dataset

In [ ]:
data.head()

In [ ]:
data.tail()

## Checking the shape of the dataset

In [ ]:
data.shape

## Checking the attribute types

In [ ]:
data.info()

## Checking the statistical summary

In [ ]:
data.describe().T

## Checking for missing values

In [ ]:
data.isnull().sum()

## Checking for duplicate values

In [ ]:
data.duplicated().sum()

# **Data Preprocessing**

In [ ]:
data["timestamp"] = pd.to_datetime(data["timestamp"], format="%m/%d/%Y %H:%M")

In [ ]:
data["dayofweek"] = data["timestamp"].dt.dayofweek

In [ ]:
data["hour"] = data["timestamp"].dt.hour

# **Exploratory Data Analysis**

## Utility Functions

In [ ]:
def histogram(data_df, col, title='Histogram', xlabel=None, ylabel='Frequency'):
    plt.figure(figsize=(9, 3.6))
    sns.histplot(data_df[col], bins=50, kde=True)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else col)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

def barchart(data_df, x_col, y_col=None, title='Bar Chart', xlabel=None, ylabel=None, rot_degrees=30):
    plt.figure(figsize=(10, 6))
    if y_col: # Bivariate bar chart (x vs y)
        sns.barplot(x=x_col, y=y_col, data=data_df)
    else: # Univariate count plot
        order = data_df[x_col].value_counts().index
        sns.countplot(data=data_df, x=x_col, order=order)

    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else ('Count' if not y_col else y_col))
    plt.xticks(rotation=rot_degrees, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def boxplot(data_df, x_col, y_col, title='Box Plot', xlabel=None, ylabel=None):
    plt.figure(figsize=(6, 3.8))
    sns.boxplot(data=data_df, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.tight_layout()
    plt.show()

def scatterplot(data_df, x_col, y_col, title='Scatter Plot', xlabel=None, ylabel=None):
    plt.figure(figsize=(6, 3.8))
    sns.scatterplot(data=data_df, x=x_col, y=y_col)
    plt.title(title)
    plt.xlabel(xlabel if xlabel else x_col)
    plt.ylabel(ylabel if ylabel else y_col)
    plt.tight_layout()
    plt.show()

## Univariate Analysis

### `failure`

In [ ]:
barchart(data, 'failure', title='Target Distribution: Drivetrain Condition', xlabel='failure (0 = normal, 1 = fault)')

### ```wind_speed_mps```

In [ ]:
histogram(data, 'wind_speed_mps', title='Distribution of wind_speed_mps', xlabel='wind_speed_mps')

### ```air_density_kgm3```

In [ ]:
histogram(data, 'air_density_kgm3', title='Distribution of air_density_kgm3', xlabel='air_density_kgm3')

### ```gearbox_oil_temp_C```

In [ ]:
histogram(data, 'gearbox_oil_temp_C', title='Distribution of gearbox_oil_temp_C', xlabel='gearbox_oil_temp_C')

### ```generator_winding_temp_C```

In [ ]:
histogram(data, 'generator_winding_temp_C', title='Distribution of generator_winding_temp_C', xlabel='generator_winding_temp_C')

### ```drivetrain_vibration_rms_mmps```

In [ ]:
histogram(data, 'drivetrain_vibration_rms_mmps', title='Distribution of drivetrain_vibration_rms_mmps', xlabel='drivetrain_vibration_rms_mmps')

### ```tower_vibration_mmps```

In [ ]:
histogram(data, 'tower_vibration_mmps', title='Distribution of tower_vibration_mmps', xlabel='tower_vibration_mmps')

### ```oil_particle_count```

In [ ]:
histogram(data, 'oil_particle_count', title='Distribution of oil_particle_count', xlabel='oil_particle_count')

### ```prior_fault_count```

In [ ]:
histogram(data, 'prior_fault_count', title='Distribution of prior_fault_count', xlabel='prior_fault_count')

### ```component_age_days```

In [ ]:
histogram(data, 'component_age_days', title='Distribution of component_age_days', xlabel='component_age_days')

## Bivariate Analysis

### ```failure``` vs ```hour```

In [ ]:
# Calculate failure rate by hour
failure_rate_by_hour_df = data.groupby('hour')['failure'].mean().reset_index()
failure_rate_by_hour_df['failure_rate_pct'] = failure_rate_by_hour_df['failure'] * 100

# Plot using the barchart utility function
barchart(data_df=failure_rate_by_hour_df, x_col='hour', y_col='failure_rate_pct',
         title='Failure Rate by Hour of Day',
         xlabel='Hour of Day', ylabel='Failure Rate (%)', rot_degrees=0)

### ```failure``` vs ```dayofweek```

In [ ]:
failure_rate_by_day_df = data.groupby('dayofweek')['failure'].mean().reset_index()
failure_rate_by_day_df['failure_rate_pct'] = failure_rate_by_day_df['failure'] * 100
# Define the chronological order of days for sorting
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# Map numerical dayofweek to names
day_name_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
failure_rate_by_day_df['dayofweek'] = failure_rate_by_day_df['dayofweek'].map(day_name_map)

# Convert 'dayofweek' to a categorical type with the defined order
failure_rate_by_day_df['dayofweek'] = pd.Categorical(failure_rate_by_day_df['dayofweek'], categories=day_order, ordered=True)

# Sort the DataFrame by the ordered 'dayofweek'
failure_rate_by_day_df = failure_rate_by_day_df.sort_values('dayofweek')

barchart(data_df=failure_rate_by_day_df, x_col='dayofweek', y_col='failure_rate_pct',
         title='Failure Rate by Day of Week',
         xlabel='Day of Week', ylabel='Failure Rate (%)', rot_degrees=30)

### ```turbine_id``` vs ```failure```

In [ ]:
barchart(data_df=data, x_col='turbine_id', y_col='failure', title='Failure Rate Across Turbines', xlabel='Turbine ID', ylabel='Failure Rate (0 = normal, 1 = fault)')

### `wind_speed_mps` vs `power_output_kW`

In [ ]:
wind_speed_bins = pd.cut(data['wind_speed_mps'], bins=20, precision=1)
power_output_mean = data.groupby(wind_speed_bins)['power_output_kW'].mean()

plt.figure(figsize=(10, 6))
plt.plot(power_output_mean.index.astype(str), power_output_mean.values, marker='o')
plt.xlabel('Wind Speed (m/s) Bins')
plt.ylabel('Average Power Output (kW)')
plt.title('Average Power Output vs. Wind Speed Bins')
plt.xticks(rotation=45, ha='right')
plt.grid(True)
plt.tight_layout()
plt.show()

### ```rotor_speed_rpm``` vs ```generator_speed_rpm```

In [ ]:
scatterplot(data_df=data, x_col='rotor_speed_rpm', y_col='generator_speed_rpm',
            title='Generator Speed vs. Rotor Speed',
            xlabel='Rotor Speed (rpm)',
            ylabel='Generator Speed (rpm)')

### ```wind_speed_mps``` vs ```rotor_speed_rpm```

In [ ]:
scatterplot(data_df=data, x_col='wind_speed_mps', y_col='rotor_speed_rpm',
            title='Rotor Speed vs Wind Speed',
            xlabel='Wind Speed (m/s)',
            ylabel='Rotor Speed (rpm)')

### `drivetrain_vibration_rms_mmps` vs `failure`

In [ ]:
boxplot(data, x_col='failure', y_col='drivetrain_vibration_rms_mmps', title='Drivetrain Vibration RMS vs Condition', xlabel='failure (0 = normal, 1 = fault)', ylabel='Drivetrain vibration RMS (mm/s)')

### `oil_particle_count` vs `failure`

In [ ]:
boxplot(data, x_col='failure', y_col='oil_particle_count', title='Oil Particle Count vs Condition', xlabel='failure (0 = normal, 1 = fault)', ylabel='Oil particle count')

### `gearbox_bearing_temp_C` vs `failure`

In [ ]:
boxplot(data, x_col='failure', y_col='gearbox_bearing_temp_C', title='Gearbox Bearing Temperature vs Condition', xlabel='failure (0 = normal, 1 = fault)', ylabel='Gearbox bearing temp (C)')

### `power_output_kW` vs `failure`

In [ ]:
boxplot(data, x_col='failure', y_col='power_output_kW', title='Power Output vs Condition', xlabel='failure (0 = normal, 1 = fault)', ylabel='Power output (kW)')

### `prior_fault_count` vs `failure`

In [ ]:
boxplot(data, x_col='failure', y_col='prior_fault_count', title='Prior Fault Count vs Condition', xlabel='failure (0 = normal, 1 = fault)', ylabel='Prior fault count')

## Multivariate Analysis

### Correlation Analysis

In [ ]:
numerical_cols = ['rated_power_kW', 'wind_speed_mps', 'wind_direction_deg', 'turbulence_intensity', 'air_density_kgm3',
                  'ambient_temp_C', 'humidity_pct', 'power_output_kW', 'rotor_speed_rpm', 'generator_speed_rpm', 'blade_pitch_angle_deg',
                  'yaw_misalignment_deg', 'gearbox_oil_temp_C', 'gearbox_bearing_temp_C', 'generator_bearing_temp_C', 'generator_winding_temp_C',
                  'main_bearing_temp_C', 'nacelle_temp_C', 'drivetrain_vibration_rms_mmps', 'tower_vibration_mmps', 'vib_fft_bearing_bpfo',
                  'vib_fft_bearing_bpfi', 'vib_fft_gearmesh', 'vib_fft_sideband', 'oil_particle_count', 'oil_pressure_bar', 'operating_hours_total',
                  'cumulative_energy_MWh', 'load_cycles', 'hours_since_last_maintenance', 'prior_fault_count', 'component_age_days']

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(data[numerical_cols].corr(), annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# **Data Preprocessing**

## Splitting the data into train, validation, and test sets

This data is **panel time-series**: 15 turbines, each logged every 10 minutes over two months, and a single degradation episode spans many consecutive rows. A random split would scatter rows from the same episode across train and test, letting the model effectively see failures it is later scored on - inflating every metric.


In [ ]:
data = data.sort_values(by=['timestamp','turbine_id'])

In [ ]:
n = len(data)

In [ ]:
# Uncomment the below train and validation indices to create a 70% train, 15% validation, and 15% test split.
# i_train = int(n * 0.70)
# i_val = int(n * 0.85)

# Uncomment the below train and validation indices to create a 80% train, 10% validation, and 10% test split.
# i_train = int(n * 0.80)
# i_val = int(n * 0.90)

# Uncomment the below train and validation indices to create a 90% train, 5% validation, and 5% test split.
# i_train = int(n * 0.90)
# i_val = int(n * 0.95)

In [ ]:
data_train = data.iloc[:i_train]
data_val = data.iloc[i_train:i_val]
data_test = data.iloc[i_val:]

X_train = data_train.drop(columns=['failure', 'timestamp', 'turbine_id'])
y_train = data_train['failure']

X_valid = data_val.drop(columns=['failure', 'timestamp', 'turbine_id'])
y_valid = data_val['failure']

X_test = data_test.drop(columns=['failure', 'timestamp', 'turbine_id'])
y_test = data_test['failure']

In [ ]:
print(f'Train split failure rate: {y_train.mean()*100:.2f}%')
print(f'Validation split failure rate: {y_valid.mean()*100:.2f}%')
print(f'Test split failure rate: {y_test.mean()*100:.2f}%')

## Missing Value Treatment

### Display the percentage of missing values

In [ ]:
print(((X_train.isnull().sum() / len(X_train) * 100).round(2)).loc[lambda x: x > 0])
print(((X_valid.isnull().sum() / len(X_valid) * 100).round(2)).loc[lambda x: x > 0])
print(((X_test.isnull().sum() / len(X_test) * 100).round(2)).loc[lambda x: x > 0])

### `gearbox_oil_temp_C`

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# oil_temp_train_stat = X_train['gearbox_oil_temp_C'].mean()    # to impute using mean
# oil_temp_train_stat = X_train['gearbox_oil_temp_C'].median()    # to impute using median
# oil_temp_train_stat = X_train['gearbox_oil_temp_C'].mode()[0]    # to impute using mode

In [ ]:
X_train['gearbox_oil_temp_C'] = X_train['gearbox_oil_temp_C'].fillna(oil_temp_train_stat)
X_valid['gearbox_oil_temp_C'] = X_valid['gearbox_oil_temp_C'].fillna(oil_temp_train_stat)
X_test['gearbox_oil_temp_C']  = X_test['gearbox_oil_temp_C'].fillna(oil_temp_train_stat)

### ```generator_bearing_temp_C```

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# bearing_temp_train_stat = X_train['generator_bearing_temp_C'].mean()    # to impute using mean
# bearing_temp_train_stat = X_train['generator_bearing_temp_C'].median()    # to impute using median
# bearing_temp_train_stat = X_train['generator_bearing_temp_C'].mode()[0]    # to impute using mode

In [ ]:
X_train['generator_bearing_temp_C'] = X_train['generator_bearing_temp_C'].fillna(bearing_temp_train_stat)
X_valid['generator_bearing_temp_C'] = X_valid['generator_bearing_temp_C'].fillna(bearing_temp_train_stat)
X_test['generator_bearing_temp_C']  = X_test['generator_bearing_temp_C'].fillna(bearing_temp_train_stat)

### ```oil_pressure_bar```

In [ ]:
# Uncomment one of the following statistics for the imputation strategy
# oil_pressure_train_stat = X_train['oil_pressure_bar'].mean()    # to impute using mean
# oil_pressure_train_stat = X_train['oil_pressure_bar'].median()    # to impute using median
# oil_pressure_train_stat = X_train['oil_pressure_bar'].mode()[0]    # to impute using mode

In [ ]:
X_train['oil_pressure_bar'] = X_train['oil_pressure_bar'].fillna(oil_pressure_train_stat)
X_valid['oil_pressure_bar'] = X_valid['oil_pressure_bar'].fillna(oil_pressure_train_stat)
X_test['oil_pressure_bar']  = X_test['oil_pressure_bar'].fillna(oil_pressure_train_stat)

# **Model Building**

**Note**: All five models provided in the subsequent subsections need to be built and evaluated.

## Model Evaluation Criterion

In [ ]:
# Uncomment one of the following evaluation metrics

# metric_of_choice = 'accuracy'
# metric_of_choice = 'precision'
# metric_of_choice = 'recall'
# metric_of_choice = 'f1'

## Utility Functions

Before moving ahead, we define a function to check the performance of the model using different metrics.

- We will be using metric functions defined in sklearn for accuracy, precision, recall, f1_score
- We will create a function which will print out all the above metrics in one go.

In [ ]:
def model_performance_classification(model, predictors, target):
    """
    Function to compute different metrics to check classification model performance

    model: classifier (sklearn or tensorflow)
    predictors: independent variables
    target: dependent variable
    """

    # 1. Get raw predictions
    pred_raw = model.predict(predictors)

    # 2. Check the shape or values to handle DL vs ML models safely
    # If the predictions are probabilities (floats between 0 and 1, not exactly 0 or 1)
    # We check if any value falls strictly between 0 and 1
    if np.issubdtype(pred_raw.dtype, np.floating) and not np.all(np.isin(pred_raw, [0.0, 1.0])):
        # FOR TENSORFLOW DL MODELS: threshold the probabilities at 0.5
        pred = (pred_raw > 0.5).astype(int)
    else:
        # FOR SKLEARN ML MODELS: use them directly
        pred = pred_raw

    # Flatten predictions to ensure they match target shape perfectly
    pred = np.array(pred).flatten()

    # 3. Compute metrics
    acc = accuracy_score(target, pred)
    recall = recall_score(target, pred)
    precision = precision_score(target, pred)
    f1 = f1_score(target, pred)

    # 4. Creating a dataframe of metrics
    data_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1},
        index=[0],
    )

    return data_perf

In [ ]:
def plot_confusion_matrix(model, predictors, target):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    # 1. Get raw predictions
    y_pred_raw = model.predict(predictors)

    # 2. Check if predictions are probabilities (typical for Keras/TF models or some ML models with predict_proba)
    # If y_pred_raw contains float values between 0 and 1, it's likely probabilities.
    if np.issubdtype(y_pred_raw.dtype, np.floating) and np.any((y_pred_raw > 0) & (y_pred_raw < 1)):
        # Binarize probabilities using a threshold (e.g., 0.5)
        y_pred = (y_pred_raw > 0.5).astype(int)
    else:
        # Use predictions directly (typical for scikit-learn classifiers that return binary labels)
        y_pred = y_pred_raw

    # Ensure y_pred is flattened to a 1D array, as confusion_matrix expects this format.
    y_pred = np.array(y_pred).flatten()

    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show() # Added plt.show() to explicitly display the plot

## Decision Tree Classifier

In [ ]:
# Set the random state for reproducibility.
# Choose an integer between 0 and 100.
# Using the same random state ensures the model produces the same results each time it is trained.
RS = __

In [ ]:
dt_class_model = DecisionTreeClassifier(class_weight='balanced', random_state=RS)

In [ ]:
dt_class_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(dt_class_model, X_train, y_train)

In [ ]:
decision_tree_class_weight_perf_train = model_performance_classification(
    dt_class_model, X_train, y_train
)
decision_tree_class_weight_perf_train

In [ ]:
plot_confusion_matrix(dt_class_model, X_valid, y_valid)

In [ ]:
decision_tree_class_weight_perf_valid = model_performance_classification(
    dt_class_model, X_valid, y_valid
)
decision_tree_class_weight_perf_valid

## Random Forest Classifier

In [ ]:
rf_class_model = RandomForestClassifier(class_weight='balanced', random_state=RS)

In [ ]:
rf_class_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(rf_class_model, X_train, y_train)

In [ ]:
random_forest_class_weight_perf_train = model_performance_classification(
    rf_class_model, X_train, y_train
)
random_forest_class_weight_perf_train

In [ ]:
plot_confusion_matrix(rf_class_model, X_valid, y_valid)

In [ ]:
random_forest_class_weight_perf_valid = model_performance_classification(
    rf_class_model, X_valid, y_valid
)
random_forest_class_weight_perf_valid

## Gradient Boosting Classifier

In [ ]:
gb_model = GradientBoostingClassifier(random_state=RS)

In [ ]:
gb_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(gb_model, X_train, y_train)

In [ ]:
gradient_boosting_weight_perf_train = model_performance_classification(
    gb_model, X_train, y_train
)
gradient_boosting_weight_perf_train

In [ ]:
plot_confusion_matrix(gb_model, X_valid, y_valid)

In [ ]:
gradient_boosting_weight_perf_valid = model_performance_classification(
    gb_model, X_valid, y_valid
)
gradient_boosting_weight_perf_valid

## XGBoost

In [ ]:
neg_count = y_train.value_counts()[0]
pos_count = y_train.value_counts()[1]
scale_pos_weight_value = neg_count / pos_count

print(f"Negative Samples: {neg_count}")
print(f"Positive Samples: {pos_count}")
print(f"Scale Pos Weight (neg/pos): {scale_pos_weight_value:.2f}")

In [ ]:
xgb_class_model = XGBClassifier(scale_pos_weight=scale_pos_weight_value, random_state=RS)

In [ ]:
xgb_class_model.fit(X_train, y_train)

In [ ]:
plot_confusion_matrix(xgb_class_model, X_train, y_train)

In [ ]:
xgb_class_weight_perf_train = model_performance_classification(
    xgb_class_model, X_train, y_train
)
xgb_class_weight_perf_train

In [ ]:
plot_confusion_matrix(xgb_class_model, X_valid, y_valid)

In [ ]:
xgb_class_weight_perf_valid = model_performance_classification(
    xgb_class_model, X_valid, y_valid
)
xgb_class_weight_perf_valid

## Neural Network (ANN)

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Choose the number of neurons for the first hidden layer between 8 and 128 integer value only
# Fewer neurons (8–32) create a simpler model that trains faster and is less likely to overfit.
# More neurons (64–128) allow the model to learn more complex patterns but increase training time and the risk of overfitting.

# Choose the activation function for the first hidden layer.
# Common choices: 'relu', 'tanh', or 'sigmoid'.

model = Sequential()
model.add(Dense(_____, activation='_____', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(7, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.summary()

In [ ]:
optimizer = 'sgd'
loss = 'binary_crossentropy'
model.compile(loss=loss, optimizer=optimizer, metrics=[tf.keras.metrics.Recall(), tf.keras.metrics.Precision(), tf.keras.metrics.BinaryAccuracy()])

In [ ]:
# Choose the number of training epochs between 10 and 100.
# Fewer epochs (10–30) train faster but may underfit the data.
# More epochs (50–100) allow the model to learn more patterns but may increase the risk of overfitting.

epochs = __

# Choose the batch size between 16 and 128.
# Smaller batch sizes (16–32) can improve generalization but increase training time.
# Larger batch sizes (64–128) train faster but may require more memory and sometimes reduce model generalization.

batch_size = __

In [ ]:
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: weights[i] for i in range(len(weights))}

In [ ]:
start = time.time()
history = model.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled,y_valid) , class_weight=class_weight_dict ,batch_size=batch_size, epochs=epochs)
end = time.time()

In [ ]:
print("Time taken in seconds ",end-start)

In [ ]:
plot_confusion_matrix(model, X_train_scaled, y_train)

In [ ]:
ann_perf_train = model_performance_classification(
    model, X_train_scaled, y_train
)
print("ANN Baseline Training Performance:")
ann_perf_train

In [ ]:
plot_confusion_matrix(model, X_valid_scaled, y_valid)

In [ ]:
ann_perf_valid = model_performance_classification(
    model, X_valid_scaled, y_valid
)
print("\nANN Baseline Validation Performance:")
ann_perf_valid

## Baseline Model Performance Comparison

### Training performance comparison

In [ ]:
print("Training performance comparison:")

all_models_train_comp_data = pd.concat([
    decision_tree_class_weight_perf_train.T.rename(columns={0: 'Decision Tree'}),
    random_forest_class_weight_perf_train.T.rename(columns={0: 'Random Forest'}),
    gradient_boosting_weight_perf_train.T.rename(columns={0: 'Gradient Boosting'}),
    xgb_class_weight_perf_train.T.rename(columns={0: 'XGBoost'}),
    ann_perf_train.T.rename(columns={0: 'Neural Network'})
], axis=1)
all_models_train_comp_data

### Validation performance comparison

In [ ]:
print("Validation performance comparison:")

all_models_valid_comp_data = pd.concat([
    decision_tree_class_weight_perf_valid.T.rename(columns={0: 'Decision Tree'}),
    random_forest_class_weight_perf_valid.T.rename(columns={0: 'Random Forest'}),
    gradient_boosting_weight_perf_valid.T.rename(columns={0: 'Gradient Boosting'}),
    xgb_class_weight_perf_valid.T.rename(columns={0: 'XGBoost'}),
    ann_perf_valid.T.rename(columns={0: 'Neural Network'})
], axis=1)
all_models_valid_comp_data

# **Hyperparameter Tuning**

**Note:** Choose at least two best-performing baseline models across the train and validation sets to proceed with tuning.

## XGBoost

In [ ]:
param_grid = {
    'n_estimators': [50, ____ , ____ , ____],          # Number of trees. Smaller values (50–100) train faster but may underfit. Larger values (100–300) improve learning but increase training time.
    'learning_rate': [0.01, ____ , ____],       # Step size for learning. Smaller values (0.01–0.05) learn gradually and need more trees. Larger values (0.1–0.3) learn faster but may overshoot.
    'max_depth': [3, ____ , ____],              # Maximum tree depth. Smaller values (3–5) reduce overfitting. Larger values (6–10) capture more complex patterns but may overfit.
    'min_child_weight': [1, ____ , ____],       # Minimum weight required for a split. Smaller values (1–3) create more complex trees. Larger values (5–10) make the model more conservative.
    'gamma': [0, 0.1, ____],             # Minimum loss reduction for a split. Smaller values (0–0.1) allow more splits. Larger values (0.3–1) reduce unnecessary splits.
    'subsample': [0.7, ____ , ____],            # Fraction of training samples per tree. Smaller values (0.6–0.8) improve generalization. Larger values (0.9–1.0) use more data but may overfit.
    'colsample_bytree': [0.4, ____ , ____],     # Fraction of features per tree. Smaller values (0.4–0.7) increase tree diversity. Larger values (0.8–1.0) use more features and may improve fit.
    'reg_alpha': [0, 0.001, ____ , ____],       # L1 regularization. Smaller values (0–0.001) apply little regularization. Larger values (0.01–0.1) reduce overfitting by encouraging sparsity.
    'reg_lambda': [0.1, ____ , ____]            # L2 regularization. Smaller values (0.1–1) apply light regularization. Larger values (5–10) provide stronger regularization.
}

In [ ]:
# Choose the number of random hyperparameter combinations to evaluate.
# Smaller values (10–20) complete faster but explore fewer parameter combinations.
# Larger values (30–50) explore a wider search space and may find better hyperparameters but increase tuning time.

n_iter = ___

xgb_randomized_cv = RandomizedSearchCV(estimator=xgb_class_model, param_distributions=param_grid, n_iter=n_iter, n_jobs = -1, scoring=metric_of_choice, cv=5, random_state=1)

xgb_randomized_cv.fit(X_train,y_train)

print("Best parameters are {} with CV score={}:" .format(xgb_randomized_cv.best_params_,xgb_randomized_cv.best_score_))

In [ ]:
tuned_xgb = xgb_randomized_cv.best_estimator_
tuned_xgb.fit(X_train, y_train)

In [ ]:
xgb_tuned_perf_train = model_performance_classification(
    tuned_xgb, X_train, y_train
)
xgb_tuned_perf_train

In [ ]:
xgb_tuned_perf_valid = model_performance_classification(
    tuned_xgb, X_valid, y_valid
)
xgb_tuned_perf_valid

## Neural Network

In [ ]:
tf.keras.backend.clear_session()

In [ ]:
model1 = Sequential()

# Choose the number of neurons for the first hidden layer between 32 and 256.
# Fewer neurons (32–64) create a simpler model that trains faster and is less likely to overfit.
# More neurons (128–256) can learn more complex patterns but increase training time and the risk of overfitting.
#
# Choose the activation function for the first hidden layer.
# Common choices are 'relu', 'tanh', or 'sigmoid'.
model1.add(Dense(___, activation='___', input_shape=(X_train_scaled.shape[1],)))

model1.add(BatchNormalization())

# Choose the dropout rate between 0.2 and 0.5.
# Lower values (0.2–0.3) retain more information but provide less regularization.
# Higher values (0.4–0.5) reduce overfitting more aggressively but may lead to underfitting if too high.
model1.add(Dropout(__))

# Choose the number of neurons for the second hidden layer between 16 and 128.
# Fewer neurons (16–32) keep the model simple and reduce overfitting.
# More neurons (64–128) increase model capacity but may overfit on smaller datasets.
#
# Choose the activation function for the second hidden layer.
# Common choices are 'relu', 'tanh', or 'sigmoid'.
model1.add(Dense(___, activation='___'))

model1.add(BatchNormalization())

# Choose the dropout rate between 0.2 and 0.5.
# Lower values (0.2–0.3) retain more information but provide less regularization.
# Higher values (0.4–0.5) reduce overfitting more aggressively but may lead to underfitting if too high.
model1.add(Dropout(___))

model1.add(Dense(1, activation='sigmoid'))

In [ ]:
model1.summary()

In [ ]:
# Uncomment one of the following optimizers.

# optimizer = 'adam'
# optimizer = 'sgd'

loss = 'binary_crossentropy'
model1.compile(loss=loss, optimizer=optimizer, metrics=[tf.keras.metrics.Recall(), tf.keras.metrics.Precision(), tf.keras.metrics.BinaryAccuracy()])

In [ ]:
# Choose the number of training epochs between 20 and 150.
# Fewer epochs (20–50) train faster but may underfit the data.
# More epochs (75–150) allow the model to learn more complex patterns but may increase the risk of overfitting.

epochs = __

# Choose the batch size between 16 and 128.
# Smaller batch sizes (16–32) update model weights more frequently and may improve generalization but increase training time.
# Larger batch sizes (64–128) train faster and use hardware more efficiently but may require more memory and sometimes reduce generalization.

batch_size = __

In [ ]:
start = time.time()
history = model1.fit(X_train_scaled, y_train, validation_data=(X_valid_scaled,y_valid) ,class_weight=class_weight_dict, batch_size=batch_size, epochs=epochs)
end = time.time()

In [ ]:
print("Time taken in seconds ",end-start)

In [ ]:
plot_confusion_matrix(model1, X_train_scaled, y_train)

In [ ]:
ann_tuned_perf_train = model_performance_classification(
    model1, X_train_scaled, y_train
)
print("ANN Tuned Training Performance:")
ann_tuned_perf_train

In [ ]:
plot_confusion_matrix(model1, X_valid_scaled, y_valid)

In [ ]:
ann_tuned_perf_valid = model_performance_classification(
    model1, X_valid_scaled, y_valid
)
print("\nANN Tuned Validation Performance:")
ann_tuned_perf_valid

## Decision Tree

In [ ]:
# Parameter grid for Decision Tree hyperparameter tuning
dt_param_grid = {
    'criterion': ['gini', 'entropy'],          # Splitting criterion. 'gini' is faster, while 'entropy' uses information gain and may produce slightly different splits.
    'max_depth': [3, ____ , ____],                    # Maximum tree depth. Smaller values (3–5) reduce overfitting. Larger values (7–15) capture more complex patterns but may overfit.
    'min_samples_split': [2, ____, ____],      # Minimum samples required to split a node. Smaller values (2–5) allow more splits. Larger values (10–20) make the tree more conservative.
    'min_samples_leaf': [1, ____, ____],       # Minimum samples required in a leaf node. Smaller values (1–3) create detailed trees. Larger values (5–10) produce smoother, more generalized trees.
    'max_features': ['sqrt', 'log2']           # Number of features considered at each split. 'sqrt' considers more features than 'log2', while 'log2' increases randomness and may reduce overfitting.
}

In [ ]:
dt_randomized_cv = RandomizedSearchCV(estimator=dt_class_model, param_distributions=dt_param_grid, n_iter=n_iter, n_jobs=-1, scoring=metric_of_choice, cv=5, random_state=1)

dt_randomized_cv.fit(X_train, y_train)  # fit the search on the training data

print("Best parameters are {} with CV score={}:".format(dt_randomized_cv.best_params_, dt_randomized_cv.best_score_))

In [ ]:
dt_randomized_cv.best_estimator_

In [ ]:
tuned_dt = DecisionTreeClassifier(dt_randomized_cv.best_estimator_)
tuned_dt.fit(X_train, y_train)

In [ ]:
dt_tuned_perf_train = model_performance_classification(
    tuned_dt, X_train, y_train
)
dt_tuned_perf_train

In [ ]:
dt_tuned_perf_valid = model_performance_classification(
    tuned_dt, X_valid, y_valid
)
dt_tuned_perf_valid

## Random Forest

In [ ]:
# Parameter grid for Random Forest hyperparameter tuning
rf_param_grid = {
    'n_estimators': [20, ____ , ____ , ____],          # Number of trees. Smaller values (20–50) train faster but may underfit. Larger values (100–300) improve stability but increase training time.
    'max_depth': [2, ____ , ____],              # Maximum tree depth. Smaller values (2–5) reduce overfitting. Larger values (7–15) capture more complex patterns but may overfit.
    'min_samples_split': [2, ____ , ____],      # Minimum samples required to split a node. Smaller values (2–5) allow more splits. Larger values (10–20) make the trees more conservative.
    'min_samples_leaf': [1, ____ , ____],       # Minimum samples required in a leaf node. Smaller values (1–3) create more detailed trees. Larger values (5–10) improve generalization.
    'max_features': ['sqrt', 'log2']           # Number of features considered at each split. 'sqrt' considers more features than 'log2', while 'log2' increases tree diversity and may reduce overfitting.
}

In [ ]:
rf_randomized_cv = RandomizedSearchCV(estimator=rf_class_model, param_distributions=rf_param_grid, n_iter=n_iter, n_jobs=-1, scoring=metric_of_choice, cv=5, random_state=1)

rf_randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(rf_randomized_cv.best_params_, rf_randomized_cv.best_score_))

In [ ]:
rf_randomized_cv.best_estimator_

In [ ]:
tuned_rf = RandomForestClassifier(rf_randomized_cv.best_estimator_)
tuned_rf.fit(X_train, y_train)

In [ ]:
rf_tuned_perf_train = model_performance_classification(
    tuned_rf, X_train, y_train
)
rf_tuned_perf_train

In [ ]:
rf_tuned_perf_valid = model_performance_classification(
    tuned_rf, X_valid, y_valid
)
rf_tuned_perf_valid

## Gradient Boosting

In [ ]:
# Parameter grid for Gradient Boosting hyperparameter tuning
gb_param_grid = {
    'n_estimators': [20, ____ , ____],      # Number of boosting stages. Smaller values (20–50) train faster but may underfit. Larger values (100–300) improve learning but increase training time.
    'learning_rate': [0.01, ____ , ____],   # Step size for learning. Smaller values (0.01–0.05) learn gradually and often require more trees. Larger values (0.1–0.3) learn faster but may overfit.
    'max_depth': [3, ____ , ____],          # Maximum depth of each tree. Smaller values (3–5) reduce overfitting. Larger values (7–10) capture more complex patterns but may overfit.
    'subsample': [0.5, ____ , ____],        # Fraction of training samples used for each boosting stage. Smaller values (0.5–0.7) improve generalization. Larger values (0.8–1.0) use more data and may improve fit but increase overfitting risk.
    'max_features': ['sqrt', 'log2']       # Number of features considered at each split. 'sqrt' considers more features than 'log2', while 'log2' increases randomness and may reduce overfitting.
}

In [ ]:
gb_randomized_cv = RandomizedSearchCV(estimator=gb_model, param_distributions=gb_param_grid, n_iter=n_iter, n_jobs=-1, scoring=metric_of_choice, cv=3, random_state=1)

gb_randomized_cv.fit(X_train, y_train)

print("Best parameters are {} with CV score={}:".format(gb_randomized_cv.best_params_, gb_randomized_cv.best_score_))

In [ ]:
gb_randomized_cv.best_estimator_

In [ ]:
tuned_gb = GradientBoostingClassifier(gb_randomized_cv.best_estimator_)
tuned_gb.fit(X_train, y_train)

In [ ]:
gb_tuned_perf_train = model_performance_classification(
    tuned_gb, X_train, y_train
)
gb_tuned_perf_train

In [ ]:
gb_tuned_perf_valid = model_performance_classification(
    tuned_gb, X_valid, y_valid
)
gb_tuned_perf_valid

* The baseline Gradient Boosting was already the best of the untuned trees on recall (**0.399**); the tune aims to push validation recall higher - a lower learning rate with more trees, plus `subsample < 1.0`, usually trades a little training fit for better generalisation.
* As always, judge it on **validation** recall (with precision as the guardrail), not the inflated training scores.

# **Final Model Selection**

The below code displays the training and validation performance comparison of the original models along with the tuned models.

In [ ]:
print("Training performance comparison:")

# Uncomment the chosen tuned models to evaluate whether hyperparameter tuning improved
# the training performance compared to the original models.

all_models_train_comp_data = pd.concat([
    all_models_train_comp_data,
    # xgb_tuned_perf_train.T.rename(columns={0: 'XGBoost (Tuned)'}),
    # ann_tuned_perf_train.T.rename(columns={0: 'Neural Network (Tuned)'}),
    # dt_tuned_perf_train.T.rename(columns={0: 'Decision Tree (Tuned)'}),
    # rf_tuned_perf_train.T.rename(columns={0: 'Random Forest (Tuned)'}),
    # gb_tuned_perf_train.T.rename(columns={0: 'Gradient Boosting (Tuned)'})
], axis=1)
all_models_train_comp_data

In [ ]:
print("Validation performance comparison")

# Uncomment the chosen tuned models to evaluate whether hyperparameter tuning improved
# the validation performance compared to the original models.

all_models_valid_comp_data = pd.concat([
    all_models_valid_comp_data,
    # xgb_tuned_perf_valid.T.rename(columns={0: 'XGBoost (Tuned)'}),
    # ann_tuned_perf_valid.T.rename(columns={0: 'Neural Network (Tuned)'}),
    # dt_tuned_perf_valid.T.rename(columns={0: 'Decision Tree (Tuned)'}),
    # rf_tuned_perf_valid.T.rename(columns={0: 'Random Forest (Tuned)'}),
    # gb_tuned_perf_valid_T.rename(columns={0: 'Gradient Boosting (Tuned)'})

], axis=1)
all_models_valid_comp_data

In [ ]:
# Uncomment the best-performing baseline or tuned model based on the evaluation metric of your choice.

# Decision Tree
# best_model = dt_class_model

# Random Forest
# best_model = rf_class_model

# Gradient Boosting
# best_model = gb_model

# XGBoost
# best_model = xgb_class_model

# Neural Network
# best_model = model

# Decision Tree (Tuned)
# best_model = tuned_dt

# Random Forest (Tuned)
# best_model = tuned_rf

# Gradient Boosting (Tuned)
# best_model = tuned_gb

# XGBoost (Tuned)
# best_model = tuned_xgb

# Neural Network (Tuned)
# best_model = model1

## Feature Importance

In [ ]:
def plot_feature_importances(model, X, y, feature_names=None, color="violet", figsize=(10, 10)):
    """
    Plots feature importances for both Traditional ML models (XGBoost, RF)
    and Deep Learning models (ANNs).

    Parameters:
    - model: Trained model object (XGBoost, Sklearn, Keras wrapper, etc.)
    - X: Evaluation features (DataFrame or 2D array)
    - y: Evaluation targets (Series or 1D array)
    - feature_names: List of feature names. If None and X is a DataFrame, uses X.columns.
    """
    # 1. Automatically grab feature names if not provided
    if feature_names is None:
        if hasattr(X, 'columns'):
            feature_names = X.columns
        else:
            feature_names = [f"Feature {i}" for i in range(X.shape[1])]

    # Convert X to numpy if it's a DataFrame for permutation calculation safety
    X_val = X.values if hasattr(X, 'columns') else X

    # 2. Extract or calculate importances
    if hasattr(model, 'feature_importances_'):
        print("💡 Detected Tree-based model. Extracting built-in feature importances...")
        importances = model.feature_importances_
        title = "Feature Importances (Tree-based Model)"
        xlabel = "Relative Importance"
    else:
        print("💡 Detected ANN/Black-box model. Calculating Permutation Importance...")
        # n_repeats is how many times a feature is shuffled; higher is more accurate but slower
        result = permutation_importance(model, X_val, y, n_repeats=5, random_state=42)
        importances = result.importances_mean
        title = "Feature Importances (Permutation Importance - ANN)"
        xlabel = "Mean Performance Drop when Shuffled"

    # 3. Sort the importances in ascending order
    indices = np.argsort(importances)

    # 4. Plotting
    plt.figure(figsize=figsize)
    plt.title(title)
    plt.barh(range(len(indices)), importances[indices], color=color, align="center")
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel(xlabel)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_feature_importances(best_model, X_train, y_train)

## Final Model Test Performance

In [ ]:
test_perf = model_performance_classification(best_model, X_test, y_test)
test_perf

In [ ]:
plot_confusion_matrix(best_model,X_test,y_test)

# **Business Insights and Recommendations**

## Business Insights

- Add to the presentation

## Recommendations

- Add to the presentation